## Resumen
Este notebook reconstruye bins de lineup para Malawi (vintage 20260922) usando world_1000bin e interpolated_means, y compara resultados contra fillgaps y GlobalDist, incluyendo ajuste p90 en la cola alta.

# Malawi: Fillgaps 

Objective: use the new `fillgaps.dta` file as the PIP/fillgaps benchmark and compute the average welfare implied by the 1000-bin distribution for Malawi in the same vintage.

Important: `fillgaps.dta` is already an aggregate PIP-style file. It does not contain microdata, so the bins are not reconstructed from `fillgaps.dta`; the 1000-bin mean is computed from the corresponding `GlobalDist1000bins` file.

Summary: We built a pipeline to reconstruct Malawi’s lineup distributions for 1997 and 2004 starting from survey-based bins in world_1000bin, scaling each source survey distribution with the interpolated-means growth factor (predicted_mean_ppp divided by svy_mean), and combining surveys using relative_distance weights; then we compared the reconstructed bins and means against both fillgaps and GlobalDist and found that reconstruction matches fillgaps means exactly while the remaining mismatch with GlobalDist is concentrated almost entirely in the very top bin, indicating the discrepancy is introduced in a later top-tail adjustment step rather than in the interpolation logic itself.

Example:

Suppose target year = 2004, using two source surveys (1997 and 2004).

Weights:

w_1997 = 0.2
w_2004 = 0.8

Scaling factors:

g_1997 = predicted_mean_ppp_1997row / svy_mean_1997 = 0.9
g_2004 = predicted_mean_ppp_2004row / svy_mean_2004 = 1.1

Source bins:

Survey 1997 bins: [10, 20, 30]
Survey 2004 bins: [8, 18, 28]

Step 1: scale each survey bin-by-bin:

1997 scaled: 0.9 × [10, 20, 30] = [9, 18, 27]
2004 scaled: 1.1 × [8, 18, 28] = [8.8, 19.8, 30.8]
Step 2: combine bin-by-bin using relative_distance:

Bin 1: 0.2×9 + 0.8×8.8 = 8.84
Bin 2: 0.2×18 + 0.8×19.8 = 19.44
Bin 3: 0.2×27 + 0.8×30.8 = 30.04

Final reconstructed bins for 2004:
[8.84, 19.44, 30.04]

## 1. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 2. Paths

In [12]:
ROOT = Path(".").resolve()
INPUT = ROOT / "01-input"
OUT = ROOT / "interpolation_excercise"
OUT.mkdir(exist_ok=True)

fillgaps_candidates = [
    INPUT / "20260922" / "lineup" / "fillgaps.dta",
    INPUT / "20260922" / "fillgaps.dta",
    INPUT / "nuevosyuki" / "fillgaps.dta",
]
fillgaps_path = next((p for p in fillgaps_candidates if p.exists()), None)

global_bins_candidates = [
    INPUT / "20260922" / "lineup" / "GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta",
    INPUT / "20260922" / "GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta",
    INPUT / "nuevosyuki" / "GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta",
]
global_bins_path = next((p for p in global_bins_candidates if p.exists()), None)

world_bins_path = INPUT / "world_1000bin.dta"
interpolated_means_path = INPUT / "interpolated_means.dta"

fillgaps_path, global_bins_path, world_bins_path, interpolated_means_path

(WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/01-input/20260922/lineup/fillgaps.dta'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/01-input/20260922/lineup/GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/01-input/world_1000bin.dta'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/01-input/interpolated_means.dta'))

## 3. Load Malawi from fillgaps

In [3]:
fillgaps = pd.read_stata(fillgaps_path, convert_categoricals=False)
fillgaps["year"] = fillgaps["year"].astype(int)

mwi_fillgaps = fillgaps[
    (fillgaps["country_code"] == "MWI")
    & (fillgaps["reporting_level"] == "national")
].copy()

mwi_fillgaps = mwi_fillgaps.sort_values("year")

mwi_fillgaps[[
    "country_code", "country_name", "year", "welfare_type", "reporting_level",
    "mean", "population", "is_interpolated", "estimate_type"
]].head(20)

,country_code,country_name,year,welfare_type,reporting_level,mean,population,is_interpolated,estimate_type
6394,MWI,Malawi,1981,1,national,5.516035,6516454,TRUE,projection
6395,MWI,Malawi,1982,1,national,5.459481,6778879,TRUE,projection
6396,MWI,Malawi,1983,1,national,5.448240,7051709,TRUE,projection
6397,MWI,Malawi,1984,1,national,5.498632,7332810,TRUE,projection
6398,MWI,Malawi,1985,1,national,5.522615,7620508,TRUE,projection
6399,MWI,Malawi,1986,1,national,5.374790,7906480,TRUE,projection
6400,MWI,Malawi,1987,1,national,5.257657,8293174,TRUE,projection
6401,MWI,Malawi,1988,1,national,5.175002,8753262,TRUE,projection
6402,MWI,Malawi,1989,1,national,5.054500,9176220,TRUE,projection
6403,MWI,Malawi,1990,1,national,5.111098,9545852,TRUE,projection


## 4. Reconstruct lineup bins from survey bins + interpolated means

This section builds Malawi lineup-like bins directly from `world_1000bin` survey distributions using `interpolated_means` weights (`relative_distance`) and scaling factors (`predicted_mean_ppp / svy_mean`).

In [9]:
if fillgaps_path is None or global_bins_path is None:
    raise FileNotFoundError("Could not find fillgaps/global bins file in expected folders.")

world_bins = pd.read_stata(world_bins_path, convert_categoricals=False)
interpolated_means = pd.read_stata(interpolated_means_path, convert_categoricals=False)
global_bins = pd.read_stata(
    global_bins_path,
    columns=["code", "year", "quantile", "welf", "pop"],
    convert_categoricals=False,
)

for df in (world_bins, interpolated_means, global_bins, fillgaps):
    df["year"] = df["year"].astype(int)

# In world_1000bin, reporting_level is numeric (1 national, 2 urban, 3 rural).
reporting_level_map = {"national": 1, "urban": 2, "rural": 3}

def reconstruct_bins(country_code: str, year: int, reporting_level: str = "national") -> pd.DataFrame:
    lvl_code = reporting_level_map[reporting_level]

    im = interpolated_means[
        (interpolated_means["country_code"] == country_code)
        & (interpolated_means["reporting_level"] == reporting_level)
        & (interpolated_means["year"] == year)
    ].copy()

    if im.empty:
        raise ValueError(f"No interpolated_means rows for {country_code} {year} ({reporting_level}).")

    # Keep a single welfare aggregate family (consumption or income) for consistency.
    target_welfare = im["welfare_type"].mode().iloc[0]
    im = im[im["welfare_type"] == target_welfare].copy()

    im["surveyid_year"] = im["surveyid_year"].astype(int)
    im["growth_factor"] = im["predicted_mean_ppp"] / im["svy_mean"]

    survey_bins = world_bins[
        (world_bins["country_code"] == country_code)
        & (world_bins["reporting_level"] == lvl_code)
        & (world_bins["welfare_type"] == target_welfare)
        & (world_bins["year"].isin(im["surveyid_year"]))
    ][["year", "percentile", "avg_welfare"]].copy().rename(columns={
        "year": "surveyid_year",
        "percentile": "bin_id",
    })

    merged = im.merge(survey_bins, on="surveyid_year", how="left")
    if merged["avg_welfare"].isna().any():
        missing_sources = sorted(merged.loc[merged["avg_welfare"].isna(), "surveyid_year"].unique())
        raise ValueError(f"Missing survey bins in world_1000bin for source years: {missing_sources}")

    merged["component"] = merged["relative_distance"] * merged["growth_factor"] * merged["avg_welfare"]

    rec = (
        merged.groupby("bin_id", as_index=False)
        .agg(
            reconstructed_welfare=("component", "sum"),
            source_rows=("surveyid_year", "size"),
        )
        .sort_values("bin_id")
    )

    pop_target = fillgaps.loc[
        (fillgaps["country_code"] == country_code)
        & (fillgaps["reporting_level"] == reporting_level)
        & (fillgaps["year"] == year),
        "population",
    ]

    if len(pop_target):
        rec["pop"] = pop_target.iloc[0] / len(rec)

    return rec

def summarize_year(country_code: str, year: int, reporting_level: str = "national") -> dict:
    rec = reconstruct_bins(country_code, year, reporting_level)
    rec_mean = rec["reconstructed_welfare"].mean()

    fg_mean = fillgaps.loc[
        (fillgaps["country_code"] == country_code)
        & (fillgaps["reporting_level"] == reporting_level)
        & (fillgaps["year"] == year),
        "mean",
    ].iloc[0]

    gd = global_bins[
        (global_bins["code"] == country_code)
        & (global_bins["year"] == year)
    ].copy()
    gd_mean = np.average(gd["welf"], weights=gd["pop"])

    return {
        "country_code": country_code,
        "year": year,
        "reconstructed_mean": rec_mean,
        "fillgaps_mean": fg_mean,
        "globaldist_mean": gd_mean,
        "diff_rec_minus_fillgaps": rec_mean - fg_mean,
        "diff_rec_minus_globaldist": rec_mean - gd_mean,
    }

summary_1997_2004 = pd.DataFrame([
    summarize_year("MWI", 1997, "national"),
    summarize_year("MWI", 2004, "national"),
]).sort_values("year")

summary_1997_2004

,country_code,year,reconstructed_mean,fillgaps_mean,globaldist_mean,diff_rec_minus_fillgaps,diff_rec_minus_globaldist
0,MWI,1997,5.724018,5.724018,4.865801,-0.000000,0.858217
1,MWI,2004,2.844316,2.844316,2.770609,-0.000000,0.073707


In [6]:
debug_im = interpolated_means[
    (interpolated_means["country_code"] == "MWI")
    & (interpolated_means["reporting_level"] == "national")
    & (interpolated_means["year"].isin([1997, 2004]))
][[
    "year", "welfare_type", "surveyid_year", "relative_distance",
    "predicted_mean_ppp", "svy_mean", "estimation_type",
    "survey_id",
]].sort_values(["year", "welfare_type", "surveyid_year"])

debug_im

,year,welfare_type,surveyid_year,relative_distance,predicted_mean_ppp,svy_mean,estimation_type,survey_id
6429,1997,consumption,1997,1.000000,5.724018,5.779660,extrapolation,MWI_1997_IHS-I_V01_M_V02_A_PIP_PC-GPWG
6442,2004,consumption,1997,0.035938,5.686365,5.779660,interpolation,MWI_1997_IHS-I_V01_M_V02_A_PIP_PC-GPWG
6443,2004,consumption,2004,0.964062,2.738372,2.740752,interpolation,MWI_2004_IHS-II_V01_M_V02_A_PIP_PC-GPWG


In [8]:
wb_debug = world_bins[
    (world_bins["country_code"] == "MWI")
    & (world_bins["reporting_level"] == 1)
    & (world_bins["year"] == 1997)
].copy()

wb_debug.shape, wb_debug[["welfare_type", "quantile", "avg_welfare"]].head(), wb_debug["avg_welfare"].mean()

((1000, 10),
         welfare_type  quantile  avg_welfare
 1689000  consumption  0.346217     0.312374
 1689001  consumption  0.377876     0.355088
 1689002  consumption  0.405986     0.395883
 1689003  consumption  0.430491     0.418235
 1689004  consumption  0.454097     0.438012,
 np.float64(5.7796597760984785))

In [10]:
rec_1997 = reconstruct_bins("MWI", 1997, "national")
rec_2004 = reconstruct_bins("MWI", 2004, "national")

rec_1997.tail(10), rec_2004.tail(10)

(          bin_id  reconstructed_welfare  source_rows           pop
 990   991.000000              26.954231            1 10,562.035000
 991   992.000000              28.631279            1 10,562.035000
 992   993.000000              31.548551            1 10,562.035000
 993   994.000000              35.805745            1 10,562.035000
 994   995.000000              38.843363            1 10,562.035000
 995   996.000000              47.171891            1 10,562.035000
 996   997.000000              57.217688            1 10,562.035000
 997   998.000000              70.958637            1 10,562.035000
 998   999.000000              83.417686            1 10,562.035000
 999 1,000.000000           2,151.780069            1 10,562.035000,
           bin_id  reconstructed_welfare  source_rows           pop
 990   991.000000              17.199013            2 12,500.737000
 991   992.000000              18.444254            2 12,500.737000
 992   993.000000              19.711317       

## 5. Bin-level diff: reconstructed lineup vs GlobalDist

Compare our reconstructed lineup bins against the actual GlobalDist bins quantile-by-quantile.
The goal is to locate **where in the distribution** the discrepancy arises for Malawi 1997 and 2004.

In [13]:
def build_bin_comparison(country_code: str, year: int, rec: pd.DataFrame) -> pd.DataFrame:
    """Align reconstructed bins with GlobalDist bins side-by-side for a given country-year."""
    gd = global_bins[
        (global_bins["code"] == country_code)
        & (global_bins["year"] == year)
    ][["quantile", "welf", "pop"]].copy().sort_values("quantile").reset_index(drop=True)

    # Assign rank 1..N so both sides align on ordinal position
    rec_aligned = rec[["bin_id", "reconstructed_welfare"]].copy().sort_values("bin_id").reset_index(drop=True)
    rec_aligned["rank"] = range(1, len(rec_aligned) + 1)

    gd["rank"] = range(1, len(gd) + 1)

    merged = rec_aligned.merge(gd, on="rank", how="inner")
    merged["diff_rec_minus_gd"] = merged["reconstructed_welfare"] - merged["welf"]
    merged["pct_diff"] = merged["diff_rec_minus_gd"] / merged["welf"]

    return merged

diff_1997 = build_bin_comparison("MWI", 1997, rec_1997)
diff_2004 = build_bin_comparison("MWI", 2004, rec_2004)

print("=== Malawi 1997: cumulative contribution of top bins to mean diff ===")
diff_1997_sorted = diff_1997.sort_values("rank")
print(f"Total mean diff (reconstructed - GlobalDist): {diff_1997_sorted['diff_rec_minus_gd'].mean():.6f}")
print(f"  Bottom 990 bins contribute: {diff_1997_sorted.iloc[:990]['diff_rec_minus_gd'].mean():.6f}")
print(f"  Top 10 bins contribute: {diff_1997_sorted.iloc[990:]['diff_rec_minus_gd'].mean():.6f}")
print(f"  Top 1 bin (bin 1000): rec={diff_1997_sorted.iloc[-1]['reconstructed_welfare']:.3f}  gd={diff_1997_sorted.iloc[-1]['welf']:.3f}")
print()
print("=== Malawi 2004: ===")
diff_2004_sorted = diff_2004.sort_values("rank")
print(f"Total mean diff: {diff_2004_sorted['diff_rec_minus_gd'].mean():.6f}")
print(f"  Bottom 990 bins: {diff_2004_sorted.iloc[:990]['diff_rec_minus_gd'].mean():.6f}")
print(f"  Top 10 bins: {diff_2004_sorted.iloc[990:]['diff_rec_minus_gd'].mean():.6f}")
print(f"  Top 1 bin (bin 1000): rec={diff_2004_sorted.iloc[-1]['reconstructed_welfare']:.3f}  gd={diff_2004_sorted.iloc[-1]['welf']:.3f}")

diff_1997[["rank", "bin_id", "reconstructed_welfare", "welf", "diff_rec_minus_gd", "pct_diff"]].tail(15)

=== Malawi 1997: cumulative contribution of top bins to mean diff ===
Total mean diff (reconstructed - GlobalDist): 0.858217
  Bottom 990 bins contribute: 0.000655
  Top 10 bins contribute: 85.756843
  Top 1 bin (bin 1000): rec=2151.780  gd=1296.486

=== Malawi 2004: ===
Total mean diff: 0.073707
  Bottom 990 bins: 0.001510
  Top 10 bins: 7.221126
  Top 1 bin (bin 1000): rec=125.599  gd=55.900


,rank,bin_id,reconstructed_welfare,welf,diff_rec_minus_gd,pct_diff
985,986,986.000000,20.771299,20.724538,0.046761,0.002256
986,987,987.000000,21.446423,21.423881,0.022542,0.001052
987,988,988.000000,21.917462,21.896685,0.020777,0.000949
988,989,989.000000,22.791365,22.744716,0.046649,0.002051
989,990,990.000000,25.138168,25.060238,0.077929,0.003110
990,991,991.000000,26.954231,26.902205,0.052026,0.001934
991,992,992.000000,28.631279,28.555092,0.076187,0.002668
992,993,993.000000,31.548551,31.416913,0.131638,0.004190
993,994,994.000000,35.805745,35.714648,0.091097,0.002551
994,995,995.000000,38.843363,38.615624,0.227739,0.005898


## 6. Apply p90 top-bin adjustment (as in lineup collapse script)

This reproduces the top-tail treatment logic:
1. Compute p90 of welfare.
2. Keep only welfare values above p90.
3. Re-estimate high-tail percentiles within that subset.
4. Replace only the top bin welfare (bin 1000) with the adjusted top-tail value.

Then compare means before/after against fillgaps and GlobalDist.

In [16]:
def apply_p90_topbin_adjustment(rec: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Apply p90-tail adjustment to top bin, mirroring the Stata collapse logic."""
    out = rec.copy().sort_values("bin_id").reset_index(drop=True)

    # Equal-pop bins imply equal weights for percentile calculations.
    p90 = np.quantile(out["reconstructed_welfare"].to_numpy(), 0.90)
    tail = out.loc[out["reconstructed_welfare"] > p90, "reconstructed_welfare"].to_numpy()

    if tail.size < 2:
        raise ValueError("Not enough bins above p90 to compute top-tail adjustment.")

    # Stata-style intent: recompute very top percentile within the p90+ tail and use it for bin 1000.
    adjusted_top = float(np.quantile(tail, 0.999))

    top_mask = out["bin_id"] == out["bin_id"].max()
    original_top = float(out.loc[top_mask, "reconstructed_welfare"].iloc[0])
    out.loc[top_mask, "reconstructed_welfare"] = adjusted_top

    stats = {
        "p90_threshold": float(p90),
        "tail_bin_count": int(tail.size),
        "original_top_bin": original_top,
        "adjusted_top_bin": adjusted_top,
        "top_bin_drop": adjusted_top - original_top,
        "mean_before": float(rec["reconstructed_welfare"].mean()),
        "mean_after": float(out["reconstructed_welfare"].mean()),
        "mean_change": float(out["reconstructed_welfare"].mean() - rec["reconstructed_welfare"].mean()),
    }
    return out, stats

rec_1997_p90, stats_1997_p90 = apply_p90_topbin_adjustment(rec_1997)
rec_2004_p90, stats_2004_p90 = apply_p90_topbin_adjustment(rec_2004)

summary_p90 = pd.DataFrame([
    {"year": 1997, **stats_1997_p90},
    {"year": 2004, **stats_2004_p90},
]).sort_values("year")

def get_reference_means(year: int) -> tuple[float, float]:
    fg = float(
        fillgaps.loc[
            (fillgaps["country_code"] == "MWI")
            & (fillgaps["reporting_level"] == "national")
            & (fillgaps["year"] == year),
            "mean",
        ].iloc[0]
    )
    gd_df = global_bins[(global_bins["code"] == "MWI") & (global_bins["year"] == year)]
    gd = float(np.average(gd_df["welf"], weights=gd_df["pop"]))
    return fg, gd

rows = []
for year, rec_raw, rec_adj in [
    (1997, rec_1997, rec_1997_p90),
    (2004, rec_2004, rec_2004_p90),
]:
    fg, gd = get_reference_means(year)
    raw_mean = float(rec_raw["reconstructed_welfare"].mean())
    adj_mean = float(rec_adj["reconstructed_welfare"].mean())
    rows.append({
        "year": year,
        "fillgaps_mean": fg,
        "globaldist_mean": gd,
        "reconstructed_mean_raw": raw_mean,
        "reconstructed_mean_p90": adj_mean,
        "diff_raw_minus_globaldist": raw_mean - gd,
        "diff_p90_minus_globaldist": adj_mean - gd,
        "diff_raw_minus_fillgaps": raw_mean - fg,
        "diff_p90_minus_fillgaps": adj_mean - fg,
    })

comparison_p90 = pd.DataFrame(rows).sort_values("year")

summary_p90, comparison_p90

(   year  p90_threshold  tail_bin_count  original_top_bin  adjusted_top_bin  \
 0  1997       6.212304             100      2,151.780069      1,947.012193   
 1  2004       4.835251             100        125.599126        116.746039   
 
    top_bin_drop  mean_before  mean_after  mean_change  
 0   -204.767876     5.724018    5.519250    -0.204768  
 1     -8.853087     2.844316    2.835463    -0.008853  ,
    year  fillgaps_mean  globaldist_mean  reconstructed_mean_raw  \
 0  1997       5.724018         4.865801                5.724018   
 1  2004       2.844316         2.770609                2.844316   
 
    reconstructed_mean_p90  diff_raw_minus_globaldist  \
 0                5.519250                   0.858217   
 1                2.835463                   0.073707   
 
    diff_p90_minus_globaldist  diff_raw_minus_fillgaps  diff_p90_minus_fillgaps  
 0                   0.653449                -0.000000                -0.204768  
 1                   0.064853                -

In [17]:
OUT.mkdir(exist_ok=True)

summary_out       = OUT / "MWI_reconstructed_lineup_mean_check_1997_2004.csv"
bins1997_out      = OUT / "MWI_reconstructed_bins_1997.csv"
bins2004_out      = OUT / "MWI_reconstructed_bins_2004.csv"
diff1997_out      = OUT / "MWI_bindiff_reconstructed_vs_globaldist_1997.csv"
diff2004_out      = OUT / "MWI_bindiff_reconstructed_vs_globaldist_2004.csv"
p90_summary_out   = OUT / "MWI_p90_adjustment_summary_1997_2004.csv"
p90_compare_out   = OUT / "MWI_p90_adjustment_mean_comparison_1997_2004.csv"
p90_bins1997_out  = OUT / "MWI_reconstructed_bins_1997_p90adjusted.csv"
p90_bins2004_out  = OUT / "MWI_reconstructed_bins_2004_p90adjusted.csv"

summary_1997_2004.to_csv(summary_out, index=False)
rec_1997.to_csv(bins1997_out, index=False)
rec_2004.to_csv(bins2004_out, index=False)
diff_1997.to_csv(diff1997_out, index=False)
diff_2004.to_csv(diff2004_out, index=False)
summary_p90.to_csv(p90_summary_out, index=False)
comparison_p90.to_csv(p90_compare_out, index=False)
rec_1997_p90.to_csv(p90_bins1997_out, index=False)
rec_2004_p90.to_csv(p90_bins2004_out, index=False)

print("Outputs saved to:", OUT)
summary_out, bins1997_out, bins2004_out, diff1997_out, diff2004_out, p90_summary_out, p90_compare_out

Outputs saved to: C:\Users\wb661551\OneDrive - WBG\Desktop\Internship\Bottom Censoring\interpolation_excercise


(WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/interpolation_excercise/MWI_reconstructed_lineup_mean_check_1997_2004.csv'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/interpolation_excercise/MWI_reconstructed_bins_1997.csv'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/interpolation_excercise/MWI_reconstructed_bins_2004.csv'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/interpolation_excercise/MWI_bindiff_reconstructed_vs_globaldist_1997.csv'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/interpolation_excercise/MWI_bindiff_reconstructed_vs_globaldist_2004.csv'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/interpolation_excercise/MWI_p90_adjustment_summary_1997_2004.csv'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/inte